# These commands are helpful to see astroquery version and relevant steuptools


In [1]:
#import pkg_resources
#print(pkg_resources.__file__)
#pip install setuptools
#import sys
#print(sys.executable)
#print(sys.version)

In [6]:
#pip install astroquery

In [11]:
#pip install spectral_cube

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [8]:
#pip install pandas

# Import Libraries

In [1]:
from astroquery.alma import Alma
import os
import numpy as np
import pandas as pd
from astroquery.alma.utils import parse_frequency_support
import astropy.units as u
from astropy.table import Table
import shutil
from pathlib import Path
import tarfile
from astropy.io import fits
from spectral_cube import SpectralCube
import matplotlib.pyplot as plt 
#import alminer
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import add_beam
from glob import glob

Matplotlib is building the font cache; this may take a moment.
Could not save font_manager cache [Errno 122] Disk quota exceeded


# Edit this for relevant molecualr line

In [4]:
sio_rest = 347.330581 * u.GHz #Splatalouge SiO (8-7)

#CO_rest = 345.7959899 * u.GHz #CO (3-2)

# These are the same keywords on the ALMA Archive: 1) Disks around low-mass stars, 2) Low mass star formation, 3) Outflows, jets, and ionized winds

In [5]:

# Give me all columns from the database. We are searching rowns with ALMA metadata
alma = Alma()
alma.archive_url = 'https://almascience.eso.org'

# Use ALMA AstroTAP query to find the dataproducts with observation keywords stored in ALMA data

In [7]:

query = """
SELECT
    obs_publisher_did,
    member_ous_uid,
    target_name,
    frequency_support,
    science_keyword
FROM ivoa.obscore
WHERE dataproduct_type = 'cube'
AND (
      science_keyword LIKE '%jets and ionized winds%'
      
    
  )
"""

# Results with all the ALMA dataproducts

In [8]:
#This results have all the data products
results = alma.query_tap(query)

In [9]:
#pd.DataFrame(results)

# Parse the frequency support in the ALMA Data

In [10]:
#Freuency support stored as string so need to parse frequncy and only need frequncy ranges
frequency_support_str = results['frequency_support'][:1]
print(frequency_support_str)
print(type(frequency_support_str))

['[217.03..217.16GHz,141.11kHz,14.5mJy/beam@10km/s,3.5mJy/beam@native, XX YY] U [218.25..218.38GHz,141.11kHz,12.7mJy/beam@10km/s,3.1mJy/beam@native, XX YY] U [219.49..219.61GHz,141.11kHz,13.9mJy/beam@10km/s,3.4mJy/beam@native, XX YY] U [220.33..220.45GHz,141.11kHz,17.6mJy/beam@10km/s,4.3mJy/beam@native, XX YY] U [230.47..230.59GHz,141.11kHz,17.7mJy/beam@10km/s,4.4mJy/beam@native, XX YY] U [231.15..231.27GHz,141.11kHz,16.3mJy/beam@10km/s,4.1mJy/beam@native, XX YY] U [231.99..233.98GHz,31250.00kHz,13.7mJy/beam@10km/s,855.9uJy/beam@native, XX YY]']
<class 'numpy.ma.MaskedArray'>


In [11]:
#Parse the frequncy support
for a in range(len(frequency_support_str)):
    freq_ranges = parse_frequency_support(frequency_support_str[a])
    print(freq_ranges,a)
    

[[217.03 217.16]
 [218.25 218.38]
 [219.49 219.61]
 [220.33 220.45]
 [230.47 230.59]
 [231.15 231.27]
 [231.99 233.98]] GHz 0


# Setup a frequency range to only find relevant data products with respect to the rest frequency

In [13]:
# Set up frequency range and in results find the range

matches = [] #Empty list
#print(type(results))
for ind,x in enumerate(results):
    #if ind >= 2:
        #continue
    ranges = parse_frequency_support(x['frequency_support'])
    #print(ranges)

    for fmin, fmax in ranges:
        if sio_rest >= fmin and sio_rest <= fmax:
            matches.append(x)
            break
print(len(matches))

1321


In [14]:
#convert results to a table
table = results.to_table()

In [15]:
#print columnames
table.colnames

['obs_publisher_did',
 'member_ous_uid',
 'target_name',
 'frequency_support',
 'science_keyword']

# Found Matches within the frequency range

In [16]:
# Convert list to a astropy.table
matches = Table(rows=matches, names=table.colnames)
#print(len(matches))

In [17]:
#print(matches)

In [18]:
#subset = 0 # only for 1 source
subset = matches[:300] #100 rows in the matches

In [19]:
#print(len(subset))
#print(subset['member_ous_uid'])

# Expand the files infromation

In [21]:
#uid = matches['member_ous_uid'][subset] # only for 1 source
uid = subset['member_ous_uid']# 300 rows from the sample


products = alma.get_data_info (uid,expand_tarfiles=True)

#print(products)

# Only selecting tar file contain science products

In [22]:

science_tar = products[
    [('001_of_001.tar' in str(x)) for x in products['access_url']]
]

#print(len(science_tar))
#print(science_tar)

# Expand tar file and only select cube data products

In [23]:
cube_files = []

for row in products:
    if 'cube' in str(row).lower() and '.fits' in str(row).lower():
        cube_files.append(row)
        print(row)

for x in cube_files:
    print(len(x))

                         ID                                                                        access_url                                                service_def error_message semantics local_semantics                     description                       content_type   content_length content_qualifier link_auth link_authorized
                                                                                                                                                                                                                                                                                           byte                                                 
--------------------------------------------------- -------------------------------------------------------------------------------------------------------- ----------- ------------- --------- --------------- --------------------------------------------------- ---------------- -------------- ----------------- --------- -----

# Download Data

In [24]:
alma.cache_location = '/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files' #save fits file to that location
download_dir = Path("/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files")
download_dir.mkdir(exist_ok=True)

# Check available columns
#print(cube_files.colnames)

# Download selected files
downloaded_files = alma.download_files(
    [row['access_url'] for row in cube_files]
)
# Move files
for f in downloaded_files:
    shutil.move(f, download_dir / Path(f).name)

INFO: Found cached file /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3621_X98e.DHNb72_sci.spw16.cube.I.mask.fits.gz with size 1517022 = 1517022. [astroquery.query]
INFO: Found cached file /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3621_X98e.DHNb72_sci.spw16.cube.I.pb.fits.gz with size 470878623 = 470878623. [astroquery.query]
INFO: Found cached file /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3621_X98e.DHNb72_sci.spw16.cube.I.pbcor.fits with size 1041436800 = 1041436800. [astroquery.query]
INFO: Found cached file /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3621_X98e.DHNb72_sci.spw18.cube.I.mask.fits.gz with size 1016006 = 1016006. [astroquery.query]
INFO: Found cached file /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_T

In [25]:
print(len(downloaded_files))

289


In [37]:
#print(downloaded_files)

In [38]:
#print(download_dir.resolve())
#print(alma.cache_location)

In [39]:
#print(download_dir.exists())
#p = Path("/users/skaur/Machine-Learning-Project/Astromorph_project/alma_astroquery/alma_TAP/scratch_files")
#print(p.exists())

In [40]:
#Alma.help_tap()

# Get Rest Frequency for all fits files

In [26]:
#download_dir = Path('/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files')

for f in download_dir.glob("*.fits"):
    if "cube" in f.name.lower():
        header = fits.getheader(f)

        print("\n", f.name)
        print("Rest frequency:", header.get("RESTFRQ", "No RESTFRQ"))


 member.uid___A001_X87c_X779.BHR71-SiO-knot_sci.spw22.cube.I.pbcor.fits
Rest frequency: 358200999000.0

 member.uid___A001_X133d_X3cb7.iras16293-2422_sci.spw23.cube.I.sd.fits
Rest frequency: 358000000000.0

 member.uid___A001_X1288_Xfa2.BHR_71_sci.spw20.cube.I.pbcor.fits
Rest frequency: 348100000000.0

 member.uid___A001_X3621_X990.DHNb72_sci.spw21.cube.I.sd.fits
Rest frequency: 347330578600.0

 member.uid___A001_X133d_X3cbd.iras16293-2422_sci.spw17.cube.I.sd.fits
Rest frequency: 345795990000.0

 member.uid___A001_X87c_X779.BHR71-SiO-knot_sci.spw16.cube.I.pbcor.fits
Rest frequency: 345795990000.0

 member.uid___A001_X133d_X3cb1.iras16293-2422_sci.spw17.cube.I.sd.fits
Rest frequency: 345795990000.0

 member.uid___A001_X133d_X3cbb.iras16293-2422_sci.spw20.cube.I.pbcor.fits
Rest frequency: 356734242000.0

 member.uid___A001_X133d_X3caf.iras16293-2422_sci.spw16.cube.I.pbcor.fits
Rest frequency: 345795990000.0

 member.uid___A001_X1288_Xf96.L1551_IRS_5_sci.spw20.cube.I.pbcor.fits
Rest freq

In [27]:
# Use rest frequency to find the right SiO cube FITS files
sio_files = []

for f in download_dir.glob("*.fits"):

    header = fits.getheader(f)

    restfreq = header.get("RESTFRQ", header.get("RESTFREQ", None))

    if restfreq is not None:

        freq1 = header["CRVAL3"] + (1 - header["CRPIX3"]) * header["CDELT3"]
        freq2 = header["CRVAL3"] + (header["NAXIS3"] - header["CRPIX3"]) * header["CDELT3"]

        freq_min_GHz = min(freq1, freq2) / 1e9
        freq_max_GHz = max(freq1, freq2) / 1e9

        if freq_min_GHz <= 347.33058 <= freq_max_GHz:
            sio_files.append(f)

In [28]:
print("SiO files:")

for f in sio_files:
    print(f)

print("Total SiO files:", len(sio_files))

SiO files:
/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1288_Xfa2.BHR_71_sci.spw20.cube.I.pbcor.fits
/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3621_X990.DHNb72_sci.spw21.cube.I.sd.fits
/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1288_Xf96.L1551_IRS_5_sci.spw20.cube.I.pbcor.fits
/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3621_X98a.DHNb71_sci.spw21.cube.I.sd.fits
/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X133d_X3cbd.iras16293-2422_sci.spw19.cube.I.sd.fits
/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X133d_X3cb1.iras16293-2422_sci.spw19.cube.I.sd.fits
/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP

In [39]:
import re

keep_files = []

# Automatically get SPWs from SiO files
sio_spws = set()

for f in sio_files:
    match = re.search(r"spw\d+", Path(f).name.lower())
    if match:
        sio_spws.add(match.group())

# Keep corresponding files
for var in all_files:
    filename = Path(var).name.lower()

    match = re.search(r"spw\d+", filename)

    if match and match.group() in sio_spws:
        if (".pb." in filename or
            ".pbcor." in filename or
            ".mask." in filename or
            ("selfcal" in filename and "pbcor" in filename)):

            keep_files.append(var)

print("SiO spectral windows:", sio_spws)
print("Total keep files:", len(keep_files))

SiO spectral windows: {'spw31', 'spw20', 'spw21', 'spw18', 'spw19', 'spw27'}
Total keep files: 108


# Look all files in the download folder

In [29]:
import glob
all_files = glob.glob("/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/*.fits*")
#all_files
#print(Path(all_files[0]).name)

In [32]:
#var = './alma_TAP/scratch_files/member.uid___A001_X3621_X98e.DHNb72_sci.spw20.cube.I.pbcor.fits'
#var = './'+str(sio_files)
#mask_file = var.split('pbcor.fits')[0]+'mask.fits.gz'
#pb_file = var.split('pbcor.fits')[0] +'pb.fits.gz'

In [33]:
#keep_files = []
#keep_files.append(var)
#keep_files.append(mask_file)
#keep_files.append(pb_file)
#print(len(keep_files))


In [40]:
keep_files_names = [Path(f).name for f in keep_files]

print("keep_files:")
for f in keep_files:
    print(repr(f))
    #print(len(f))


print("\nall_files:")
for f in all_files:
    print(repr(f))

keep_files:
'/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3621_X98e.DHNb72_sci.spw20.cube.I.pb.fits.gz'
'/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3788_X5e2f.08303-4303_sci.spw31.cube.I.selfcal.pb.fits.gz'
'/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X133d_X3caf.iras16293-2422_sci.spw20.cube.I.mask.fits.gz'
'/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1288_Xfa2.BHR_71_sci.spw20.cube.I.pbcor.fits'
'/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X133d_X3ca9.iras16293-2422_sci.spw20.cube.I.pb.fits.gz'
'/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X133d_X3cb5.iras16293-2422_sci.spw20.cube.I.pb.fits.gz'
'/users/skaur/lustre/Machine

## Move SiO Fits files in the final folder

In [42]:
for y in all_files:
    if Path(y).name in keep_files_names :
        destination = Path(y.replace('scratch_files', 'final_fits_file'))
        #print(destination)
        # Make sure destination directory exists
        destination.parent.mkdir(parents=True, exist_ok=True)

        shutil.move(Path(y), destination)
        print("Moved:", y)

Moved: /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3621_X98e.DHNb72_sci.spw20.cube.I.pb.fits.gz
Moved: /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3788_X5e2f.08303-4303_sci.spw31.cube.I.selfcal.pb.fits.gz
Moved: /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X133d_X3caf.iras16293-2422_sci.spw20.cube.I.mask.fits.gz
Moved: /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1288_Xfa2.BHR_71_sci.spw20.cube.I.pbcor.fits
Moved: /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X133d_X3ca9.iras16293-2422_sci.spw20.cube.I.pb.fits.gz
Moved: /users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X133d_X3cb5.iras16293-2422_sci.spw20.cube.I.pb.fits.gz
Moved: /us

## Delete Fits files have no SiO lines

In [103]:
remaining_files = glob.glob('./alma_TAP/scratch_files/*')

for y in remaining_files:
    os.remove(y)
    print("Deleted:", y)
print(len(remaining_files))

Deleted: ./alma_TAP/scratch_files/member.uid___A001_X3621_X98e.DHNb72_sci.spw20.cube.I.pb.fits.gz
Deleted: ./alma_TAP/scratch_files/706acbfba3161331421e10cafb68b6c9a2ca6ad1b5a69795a88b1f11.pickle
Deleted: ./alma_TAP/scratch_files/e654a1d727ad53499bdd93cbbb26f2d1918b7189d02eb3a62ca68ee2.pickle
Deleted: ./alma_TAP/scratch_files/member.uid___A001_X3788_X5e2f.08303-4303_sci.spw31.cube.I.selfcal.pb.fits.gz
Deleted: ./alma_TAP/scratch_files/cd2222ee59840391e6a22630193bcc9368bf511373efc0d0de179499.pickle
Deleted: ./alma_TAP/scratch_files/d51e9861a9ede027cef3aa0870bc73a8e44f96f5b880aeceeef26714.pickle
Deleted: ./alma_TAP/scratch_files/e3168bf512bd7c98409396c59a12e6bf2940c3c15c5136a80d0a61fa.pickle
Deleted: ./alma_TAP/scratch_files/6d80e9e4aff2d39ab095222ee689006cf0e85268aabc267c3c9ec783.pickle
Deleted: ./alma_TAP/scratch_files/f0fc37dbe7fae78f2b649666365d2072f2efaf50757fb168dde6abb0.pickle
Deleted: ./alma_TAP/scratch_files/d95bd9438bd4524a9eb8b067d75585c72f2a0fd8aa91fa92cf31c8ee.pickle
Deleted

# Open SiO fits files from the folder

In [45]:
# Folder containing the final FITS files
final_dir = Path("./alma_TAP/final_fits_file")

# Find the pbcor FITS file
pbcor_files = list(final_dir.glob("*pbcor.fits"))

#print("Found:", pbcor_files)
#print(len(pbcor_files))